# RescueGroups API Debugger

Tests RescueGroups v5 public API for pet adoption data. Verifies:
1. API key works
2. Search by location returns animals
3. Each animal has a `url` field that resolves to a real adoption page
4. Photos, description, breed, age are all populated

If all 4 pass, we replace Petfinder scraping with this API.

## 1. Config

Paste your RescueGroups API key here OR set `RESCUE_GROUP_API_KEY` in your shell env first.

In [ ]:
import os

API_KEY = os.environ.get('RESCUE_GROUP_API_KEY', '')
if not API_KEY:
    API_KEY = input('Paste your RescueGroups API key: ').strip()

API_BASE = 'https://api.rescuegroups.org/v5/public'

# Test locations from your newsletter config
TEST_LOCATIONS = [
    {'name': 'East Cobb',  'postalcode': '30062', 'radius_miles': 25},
    {'name': 'Perimeter',  'postalcode': '30346', 'radius_miles': 25},
]

print(f'API key loaded ({len(API_KEY)} chars)')
print(f'Locations: {[l["name"] for l in TEST_LOCATIONS]}')

## 2. Search for available cats near East Cobb

Verifies the API key works and returns animals.

In [ ]:
import requests, json

loc = TEST_LOCATIONS[0]

payload = {
    'data': {
        'filterRadius': {'miles': loc['radius_miles'], 'postalcode': loc['postalcode']},
        'filters': [
            {'fieldName': 'species.singular', 'operation': 'equals', 'criteria': 'Cat'},
            {'fieldName': 'statuses.name',    'operation': 'equals', 'criteria': 'Available'},
        ],
    },
    'limit': 25,
}

r = requests.post(
    f'{API_BASE}/animals/search/available',
    headers={'Authorization': API_KEY, 'Content-Type': 'application/vnd.api+json'},
    json=payload,
    timeout=20,
)
print(f'Status: {r.status_code}')
if r.status_code != 200:
    print(f'Body (first 500): {r.text[:500]}')
else:
    body = r.json()
    animals = body.get('data', [])
    print(f'Got {len(animals)} animals near {loc["name"]}')

## 3. Inspect the first animal's structure

Show every attribute so we know exactly what we have to work with.

In [ ]:
if r.status_code == 200 and animals:
    a = animals[0]
    print(f'Top-level keys: {list(a.keys())}\n')
    attr = a.get('attributes', {})
    print(f'attributes keys ({len(attr)} total):')
    for k in sorted(attr.keys()):
        v = attr[k]
        t = type(v).__name__
        if v is None:
            preview = 'null'
        elif isinstance(v, list):
            preview = f'list[{len(v)}]'
        elif isinstance(v, dict):
            preview = f'dict (keys: {list(v.keys())[:6]})'
        elif isinstance(v, str):
            preview = repr(v[:80] + ('…' if len(v) > 80 else ''))
        else:
            preview = repr(v)
        print(f'  {k:35s} {t:10s} {preview}')

## 4. Pull the fields we actually need for the newsletter

Maps RescueGroups → newsletter pet schema.

In [ ]:
if r.status_code == 200:
    print(f'=== {len(animals)} animals near {loc["name"]} ===\n')
    for a in animals[:8]:
        attr = a.get('attributes', {})
        print(f"Name:       {attr.get('name')}")
        print(f"Species:    {attr.get('speciesSingular') or attr.get('species', {})}")
        print(f"Breed:      {attr.get('breedString') or attr.get('breedPrimary')}")
        print(f"Age:        {attr.get('ageGroup')}  ({attr.get('ageString')})")
        print(f"Sex:        {attr.get('sex')}")
        print(f"Size:       {attr.get('sizeGroup')}")
        # Description fields — try a few names
        desc = attr.get('descriptionText') or attr.get('description') or ''
        print(f"Desc len:   {len(desc)}")
        print(f"Desc 1st:   {desc[:160]}")
        # Picture count
        pics = attr.get('pictures') or []
        print(f"Pictures:   {len(pics)}")
        # THE adoption URL
        print(f"URL:        {attr.get('url')}")
        print('-' * 70)

## 5. Verify the URL actually loads (HTTP 200)

The whole point of this exercise — confirm the URL goes to a real, viewable adoption page.

In [ ]:
from urllib.parse import urlparse

if r.status_code == 200:
    print('Verifying the first 5 URLs return HTTP 200…\n')
    for a in animals[:5]:
        attr = a.get('attributes', {})
        url = attr.get('url')
        name = attr.get('name')
        if not url:
            print(f'  ⚠ {name}: NO url field')
            continue
        try:
            chk = requests.get(url, timeout=10, allow_redirects=True,
                               headers={'User-Agent': 'Mozilla/5.0'})
            host = urlparse(chk.url).netloc
            print(f'  {chk.status_code} | {name:30s} → {host}{urlparse(chk.url).path[:60]}')
        except Exception as e:
            print(f'  ✗ {name}: {e}')

## 6. Inspect picture structure

We need 3 photos per pet for the GIF. Check what the API gives us.

In [ ]:
if r.status_code == 200 and animals:
    # Find one with multiple pics
    pet_with_pics = next((a for a in animals if (a.get('attributes', {}).get('pictures') or [])), None)
    if not pet_with_pics:
        print('No pets with pictures found in this batch.')
    else:
        attr = pet_with_pics.get('attributes', {})
        pics = attr.get('pictures') or []
        print(f"{attr.get('name')}: {len(pics)} pictures\n")
        for i, p in enumerate(pics[:5]):
            if isinstance(p, dict):
                print(f'  Picture {i}: type=dict, keys={list(p.keys())}')
                # Print a few likely URL fields
                for k in ('large', 'original', 'url', 'urlSecureFullsize', 'urlSecureThumbnail', 'fileSize'):
                    if k in p:
                        v = p[k]
                        s = str(v)[:120] if v else 'null'
                        print(f'    {k}: {s}')
            else:
                print(f'  Picture {i}: {type(p).__name__} = {str(p)[:120]}')

## 7. Shelter / org info — needed for the newsletter

The newsletter footer for each pet shows shelter name, address, phone. Check what's available.

In [ ]:
if r.status_code == 200 and animals:
    a = animals[0]
    attr = a.get('attributes', {})
    rels = a.get('relationships', {})
    print('Animal-level org/shelter fields on this pet:')
    for k in sorted(attr.keys()):
        if 'org' in k.lower() or 'rescue' in k.lower() or 'shelter' in k.lower() or 'contact' in k.lower():
            v = attr[k]
            print(f'  {k}: {str(v)[:120]}')
    print(f'\nRelationships keys: {list(rels.keys())}')
    # If 'orgs' or 'organizations' is a relationship, grab the included data
    included = body.get('included', [])
    if included:
        orgs = [i for i in included if i.get('type') in ('orgs', 'organizations')]
        if orgs:
            print(f'\nincluded orgs ({len(orgs)}). First org attributes:')
            o_attr = orgs[0].get('attributes', {})
            for k in ('name', 'addressLine1', 'city', 'state', 'postalcode', 'phone', 'email', 'url'):
                if k in o_attr:
                    print(f'  {k}: {o_attr[k]}')

## 8. Verdict — is this enough to replace Petfinder?

Run all cells above, then check this list:

- [ ] Cell 2 returned 200 + animals
- [ ] Cell 4 shows real names, breeds, ages, descriptions (not empty)
- [ ] Cell 4 shows `pictures > 0` for at least most pets
- [ ] Cell 5 shows the `url` field returns HTTP 200 (page loads)
- [ ] Cell 6 shows pictures have a usable URL field (large / original / urlSecureFullsize)
- [ ] Cell 7 shows shelter name + contact info available somewhere

If all 6 pass → migration is straightforward. Tell me and I'll rewrite `Furry_Friends_Marietta.py` to use this API instead of scraping Petfinder.